# Exploratory Data Analysis (EDA): Austrian Academic Labour Well-being

This notebook performs a detailed exploratory analysis focused specifically on **Academic Staff in Austria** using the **final processed dataset** (`labour_wellbeing_processed_v1.csv`). The primary goal is to prepare the data and gain insights for subsequent Structural Equation Modeling (SEM) focused on explaining **Burnout_Score** within this specific subgroup.

**Objectives:**
1.  **Load and Prepare:** Load the full processed data, apply necessary label encoding, and then **filter** the dataset to retain only Austrian academic staff.
2.  **Univariate Analysis:** Understand the distribution and characteristics of *all* relevant variables within the filtered subset.
3.  **Multivariate Analysis (Correlations):** Explore relationships using:
    * Scatter plot matrices (Pairplots) for key numerical variables (Work Conditions, Motivations (WM_), Attitudes (VB_)) related to `Burnout_Score` within the subset.
    * Full Pearson correlation matrix for quantitative variables in the subset.
    * Grouped correlation matrices by theoretical domains focusing on `Burnout_Score` within the subset.
4.  **Bivariate Analysis (Categorical vs. Numerical):** Visualize how numerical variable distributions change across relevant categories (e.g., Gender, Age Group, Subject Area) within the subset.
5.  **Disparity Analysis (Compact):** Investigate potential differences in key well-being metrics across demographic/institutional groups within the subset, using grouped statistics and **compact subplot visualizations**.
6.  **SEM Preparation:** Identify potential multicollinearity and understand variable distributions specific to Austrian academics to inform SEM model specification.

## 1. Setup: Import Libraries

In [ ]:
# Core Libraries for Data Manipulation
import pandas as pd
import numpy as np
import math # For calculating subplot grid size

# Libraries for Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde # For smooth density lines in histograms

# Utilities
from IPython.display import display, Markdown # For displaying outputs nicely in Jupyter
import warnings # To manage warnings

# --- Configuration ---
warnings.filterwarnings('ignore') # Suppress routine warnings for cleaner output
sns.set_theme(style="whitegrid", palette="muted") # Set consistent plot theme
plt.rcParams['figure.figsize'] = (14, 6) # Default figure size
plt.rcParams['axes.titlesize'] = 16 # Title font size
plt.rcParams['axes.labelsize'] = 12 # Axis label font size
plt.rcParams['xtick.labelsize'] = 10 # X-tick label size
plt.rcParams['ytick.labelsize'] = 10 # Y-tick label size
plt.rcParams['figure.autolayout'] = True # Enable auto layout to prevent overlaps
pd.set_option('display.max_columns', None) # Show all columns in DataFrames
pd.set_option('display.width', 1000) # Adjust display width

## 2. Data Loading

Load the full processed dataset generated by `data_preparation.ipynb`.

In [ ]:
# Specify the path to your final processed dataset
# This should be the output from the data_preparation notebook
file_path = 'https://raw.githubusercontent.com/EduardoAve/Labour-well-being/main/data/02_prepared/Final_Dataset_Processed.csv'

try:
    # Attempt to load the dataset
    df_full_raw = pd.read_csv(file_path)
    print(f"Full dataset '{file_path}' loaded successfully.")
    print(f"Original Dimensions: {df_full_raw.shape}")
    display(df_full_raw.head())
except FileNotFoundError:
    print(f"Error: File '{file_path}' not found. Please ensure the file is in the correct directory or provide the full path.")
    df_full_raw = pd.DataFrame() # Create an empty DataFrame to prevent subsequent errors
except Exception as e:
    print(f"An error occurred while loading the data: {e}")
    df_full_raw = pd.DataFrame()

## 3. Initial Data Check (Full Dataset)

Verify data types and missing values on the full dataset before filtering.

In [ ]:
if not df_full_raw.empty:
    print("\n--- General Information and Data Types (Full Dataset) ---")
    df_full_raw.info()

    print("\n--- Missing Values Summary (Full Dataset) ---")
    missing_values = df_full_raw.isnull().sum()
    missing_percentage = (missing_values / len(df_full_raw)) * 100
    missing_info = pd.DataFrame({'Count': missing_values, 'Percentage': missing_percentage})
    missing_info = missing_info[missing_info['Count'] > 0]
    if not missing_info.empty:
        print("Missing values found in the loaded dataset:")
        display(missing_info.sort_values(by='Count', ascending=False))
    else:
        print("No missing values initially found in the loaded dataset.")

    # Create a working copy for preprocessing the full dataset first
    df_full_cleaned = df_full_raw.copy()

else:
    print("Full DataFrame is empty. Further analysis cannot proceed.")
    df_full_cleaned = pd.DataFrame() # Ensure df_full_cleaned exists even if empty

## 4. Data Preparation for EDA (Labeling and Filtering)

Steps:
1.  **Label Encoding:** Convert numerical categorical variables to meaningful text labels on the full dataset.
2.  **Create Age Groups:** Bin the 'Age' variable.
3.  **Filter Data:** Select only Academic Staff from Austria.

### 4.1 Label Encoding for Categorical Variables (on Full Dataset)

Apply mappings to create human-readable labels. This is done on the full dataset first to ensure consistent labels are available before filtering.

In [ ]:
if not df_full_cleaned.empty:
    print("Applying label encoding to categorical variables...")
    # --- Define Mappings (CRITICAL: Verify these match your actual data codes and desired labels) ---
    country_map = {1: 'Austria', 2: 'Czech Republic'} # Assuming 1=AT, 2=CZ
    gender_map = {1.0: 'Male', 2.0: 'Female', 3.0: 'Other'}
    marital_map = {
        1.0: 'Married/Registered Partnership', 2.0: 'In a relationship (unmarried)',
        3.0: 'Single', 4.0: 'Divorced', 5.0: 'Widowed', 6.0: 'Other_Marital'
    }
    care_map = {
        1.0: 'No', 2.0: 'Care for underage children', 3.0: 'Care for dependent relatives',
        4.0: 'Combination Care'
    }
    institution_type_map = {
        1.0: 'CZ: Public HEI', 2.0: 'CZ: Private HEI', 3.0: 'CZ: State HEI',
        4.0: 'AT: Public University', 5.0: 'AT: Private University/College',
        6.0: 'AT: University of Applied Sciences',
        7.0: 'AT: Public Uni College Teacher Ed', 8.0: 'AT: Private Uni College Teacher Ed'
    }
    subject_area_map = {
        1.0: 'Natural sciences', 2.0: 'Technical sciences', 3.0: 'Agricultural/forestry/veterinary',
        4.0: 'Healthcare/medical/pharmaceutical', 5.0: 'Humanities/social sciences',
        6.0: 'Economic sciences', 7.0: 'Law', 8.0: 'Pedagogy/teacher training',
        9.0: 'Culture/art', 10.0: 'Sport sciences', 11.0: 'Unspecified/cannot be categorised',
        12.0: 'Security/defence/Military', 13.0: 'Other_Faculty'
    }
    contract_duration_map = {
        1.0: 'Permanent/Continuous (CZ/AT)', 2.0: 'Fixed-term (permanent prospects) (CZ/AT)',
        3.0: 'Fixed-term (no permanent prospects) (CZ/AT)', 4.0: 'Casual/hourly (CZ/AT)',
        5.0: 'Fixed-term (unspecified prospects AT?)', 6.0: 'Permanent (tenured AT?)',
        7.0: 'Other_Contract'
    }
    effort_comparison_map = {1.0: 'Equal', 2.0: 'Less', 3.0: 'More'}
    holds_leadership_map = {
        1.0: 'No', 2.0: 'Yes (Institution/Faculty/Dept)', 3.0: 'Yes (Research Team)',
        4.0: 'Combination Leadership'
    }
    policy_influence_map = {1.0: '1: Not influential', 2.0: '2', 3.0: '3', 4.0: '4', 5.0: '5: Very influential'}
    academic_non_academic_map = {1: 'Non-academic', 2: 'Academic'}
    current_position_map = { # Example - VERIFY CODES
        2.0: 'Lecturer (CZ/AT Lector)', 3.0: 'Assistant (CZ)', 4.0: 'Assistant professor (CZ)',
        5.0: 'Docent (CZ)', 6.0: 'Professor (CZ)', 7.0: 'Researcher (CZ)',
        8.0: 'Externist (CZ)', 9.0: 'Researcher & Academic (CZ)',
        10.0: 'Postdoc Assistant (AT)', 11.0: 'Assistant Prof (AT)', 12.0: 'Associate Prof (AT)',
        13.0: 'University Prof (AT)', 14.0: 'Senior Scientist/Lecturer (AT)', 15.0: 'Project Staff (AT)',
        16.0: 'Other Position (AT)', 17.0: 'Student Assistant (AT)',
        1.0: 'Non-academic Role (Generic)', 0.0: 'Unknown/NA Position'
    }
    job_description_map = { # For Job_Description_Category (Non-academic)
        1.0: 'Dept/manager assistant', 2.0: 'Lab technician', 3.0: 'Librarian/archivist',
        4.0: 'Facility management', 5.0: 'ICT', 6.0: 'Student support',
        7.0: 'Economics/finance/HR', 8.0: 'Project management', 9.0: 'Legal/control',
        10.0: 'Marketing/PR', 11.0: 'Science/knowledge transfer', 12.0: 'Foreign affairs',
        13.0: 'Other administrative', 14.0: 'Other/Combination Admin'
    }
    education_level_map = { # For Education_Level
        1.0: 'Elementary', 2.0: 'Apprenticeship', 3.0: 'Vocational/Commercial School',
        4.0: 'High school/Secondary', 5.0: 'Higher professional school (CZ)',
        6.0: 'Bachelor', 7.0: 'Master', 8.0: 'Doctoral/PhD'
    }
    has_other_job_map = {
        1.0: 'No', 2.0: 'Yes (Public Sector)', 3.0: 'Yes (Private Sector)',
        4.0: 'Yes (Non-profit)', 5.0: 'Yes (Self-employed)',
        6.0: 'Yes (Multiple Areas)', 7.0: 'Yes (Other_Job)'
    }

    # Dictionary linking final column names to their maps
    mappings_to_apply = {
        'Country': country_map,
        'Gender': gender_map,
        'Marital_Status': marital_map,
        'Cares_for_Dependents': care_map,
        'Institution_Type': institution_type_map,
        'Subject_Area': subject_area_map,
        'Contract_Duration': contract_duration_map,
        'Effort_Comparison': effort_comparison_map,
        'Holds_Leadership_Position': holds_leadership_map,
        'Policy_Influence': policy_influence_map,
        'Academic/Non-academic': academic_non_academic_map,
        'Current_Position': current_position_map,
        'Job_Description_Category': job_description_map,
        'Education_Level': education_level_map,
        'Has_Other_Job': has_other_job_map
    }

    labeled_cols = [] # Keep track of newly created label columns

    for col, mapping in mappings_to_apply.items():
        if col in df_full_cleaned.columns:
            label_col = f"{col}_Label" # Create new column name
            df_full_cleaned[label_col] = df_full_cleaned[col].map(mapping)
            # Convert the new labeled column to categorical dtype
            is_ordered = (col == 'Policy_Influence' or col == 'Effort_Comparison' or col == 'Education_Level')
            try:
                all_categories = sorted(list(set(mapping.values()))) if not is_ordered else list(mapping.values())
                df_full_cleaned[label_col] = pd.Categorical(df_full_cleaned[label_col], categories=all_categories, ordered=is_ordered)
                print(f"  - Labeled column '{label_col}' created for '{col}'. Type: {'Ordinal' if is_ordered else 'Nominal'}")
                labeled_cols.append(label_col)
            except Exception as e:
                print(f"  - Warning: Could not convert '{label_col}' to categorical. Error: {e}")
                if label_col in df_full_cleaned.columns:
                    df_full_cleaned[label_col] = df_full_cleaned[label_col].astype('object')
        else:
            print(f"  - Warning: Column '{col}' not found in DataFrame, skipping mapping.")

    print("\nLabel encoding completed on full dataset.")

else:
    print("Full DataFrame is empty, label encoding skipped.")
    df_full_cleaned = pd.DataFrame()

### 4.2 Create Age Groups (on Full Dataset)

In [ ]:
if not df_full_cleaned.empty:
    # Define age bins and labels
    age_bins = [0, 29.9, 39.9, 49.9, 59.9, np.inf]
    age_labels = ['<30', '30-39', '40-49', '50-59', '60+']
    if 'Age' in df_full_cleaned.columns:
        df_full_cleaned['Age_Group'] = pd.cut(df_full_cleaned['Age'], bins=age_bins, labels=age_labels, right=True)
        # Ensure it's treated as an ordered categorical variable
        df_full_cleaned['Age_Group'] = pd.Categorical(df_full_cleaned['Age_Group'], categories=age_labels, ordered=True)
        print("'Age_Group' column created successfully.")
        # Add Age_Group to the list of labeled categorical columns if not already there
        if 'labeled_cols' in locals() and 'Age_Group' not in labeled_cols:
            labeled_cols.append('Age_Group')
    else:
        print("Warning: 'Age' column not found, could not create 'Age_Group'.")
else:
    print("Empty DataFrame, age group creation skipped.")

### 4.3 Filter Data for Austrian Academic Staff

Create the final DataFrame `df_analysis` containing only the relevant subset for the SEM.

In [ ]:
df_analysis = pd.DataFrame() # Initialize empty dataframe

if not df_full_cleaned.empty:
    # Define filter conditions using the LABEL columns
    country_filter = df_full_cleaned['Country_Label'] == 'Austria'
    role_filter = df_full_cleaned['Academic/Non-academic_Label'] == 'Academic'

    # Apply filters
    df_analysis = df_full_cleaned[country_filter & role_filter].copy()

    # Drop columns that might be constant after filtering or irrelevant for academics
    cols_to_drop_after_filter = ['Country', 'Country_Label', 'Academic/Non-academic', 'Academic/Non-academic_Label',
                                 'Job_Description_Category', 'Job_Description_Category_Label'] # Job Desc is for non-academics
    # Check which columns actually exist before dropping
    cols_to_drop_existing = [col for col in cols_to_drop_after_filter if col in df_analysis.columns]
    df_analysis.drop(columns=cols_to_drop_existing, inplace=True)

    print(f"Filtered dataset created for Austrian Academic Staff.")
    print(f"Filtered Dimensions: {df_analysis.shape}")
    if df_analysis.empty:
        print("Warning: The filtered dataset is empty. Please check filter conditions and source data.")
    else:
        display(df_analysis.head())

        # Update the list of categorical columns for the filtered data
        if 'labeled_cols' in locals():
            labeled_cols = [col for col in labeled_cols if col in df_analysis.columns]

else:
    print("Full cleaned DataFrame is empty, filtering skipped.")

## 5. Exploratory Data Analysis (EDA) - Austrian Academic Staff

Analyze distributions, relationships, and potential disparities in the filtered data (`df_analysis`).

### 5.1 Univariate Analysis (Filtered Data)

Examine the distribution of each variable within the Austrian academic subset.

In [ ]:
if not df_analysis.empty:
    # --- Identify final numeric and categorical columns for analysis (in the filtered df) ---

    # Define conceptually categorical columns (original names, excluding dropped ones)
    categorical_original_names_filtered = [
         'Gender', 'Marital_Status', 'Cares_for_Dependents',
         'Institution_Type', 'Subject_Area', 'Contract_Duration',
         'Effort_Comparison', 'Holds_Leadership_Position', 'Policy_Influence', 'Has_Other_Job',
         'Current_Position', 'Education_Level'
    ]
    cols_to_exclude_from_numeric_filtered = ['Version'] + categorical_original_names_filtered

    # Numeric columns are those with number dtype excluding original categorical codes
    all_numeric_df_cols_filtered = df_analysis.select_dtypes(include=np.number).columns
    numeric_cols_for_analysis_filtered = all_numeric_df_cols_filtered.difference(cols_to_exclude_from_numeric_filtered, sort=False)

    # Categorical columns are the ones ending in _Label or Age_Group (in the filtered df)
    categorical_cols_for_analysis_filtered = [col for col in df_analysis.columns if col.endswith('_Label') or col == 'Age_Group']

    # --- Numerical Descriptive Statistics ---
    display(Markdown("#### Descriptive Statistics (Numerical Variables - Filtered)"))
    if not numeric_cols_for_analysis_filtered.empty:
        display(df_analysis[numeric_cols_for_analysis_filtered].describe().T.round(2))
    else:
        print("No numerical columns found for descriptive statistics in the filtered dataset.")

    # --- Categorical Frequencies ---
    display(Markdown("#### Frequencies (Categorical Variables - Labeled & Filtered)"))
    if categorical_cols_for_analysis_filtered:
        for col in categorical_cols_for_analysis_filtered:
            if col in df_analysis.columns:
                # Clean up the column name for display
                display_col_name = col.replace('_Label','').replace('_',' ')
                display(Markdown(f"##### {display_col_name}"))
                # Calculate frequency and percentage
                freq_table = df_analysis[col].value_counts(dropna=False).to_frame(name="Frequency") # Include NaNs
                freq_table['Percentage'] = (df_analysis[col].value_counts(normalize=True, dropna=False) * 100).round(2)
                display(freq_table)
            else:
                print(f"Warning: Column {col} not found for frequency count in filtered data.")
    else:
        print("No labeled categorical columns found for frequency counts in filtered data.")

else:
    print("Filtered DataFrame is empty, univariate statistics skipped.")

In [ ]:
if not df_analysis.empty:
    display(Markdown("#### Univariate Visualizations (Filtered Data)"))

    # Define consistent palette and figure size
    uni_palette = "viridis"
    fig_size_uni = (14, 5)

    # --- Plots for Numerical Variables ---
    if not numeric_cols_for_analysis_filtered.empty:
        display(Markdown("##### Numerical Distributions (Histograms & Boxplots - Filtered)"))
        for col in numeric_cols_for_analysis_filtered:
            # Clean up column name for title
            clean_col_name = col.replace('_', ' ')
            try:
                # Check if there is data to plot
                if df_analysis[col].notna().sum() < 2: # Need at least 2 non-NA points
                    continue

                fig, axes = plt.subplots(1, 2, figsize=fig_size_uni)
                fig.suptitle(f'Distribution of {clean_col_name} (Austrian Academics)', fontsize=16, y=1.03)

                # Histogram with KDE
                sns.histplot(df_analysis[col], kde=True, ax=axes[0], bins=30, color=sns.color_palette(uni_palette, 2)[0])
                axes[0].set_title('Histogram & Density')
                axes[0].set_xlabel(clean_col_name)
                axes[0].set_ylabel('Frequency / Density')

                # Boxplot
                sns.boxplot(x=df_analysis[col], ax=axes[1], color=sns.color_palette(uni_palette, 2)[1])
                axes[1].set_title('Boxplot')
                axes[1].set_xlabel(clean_col_name)

                plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust layout
                plt.show()
            except Exception as e:
                print(f"Could not plot numerical variable {col}. Error: {e}")
                if plt.gcf().get_axes(): plt.close()
    else:
        print("No numerical columns found for plotting in the filtered dataset.")

    # --- Plots for Categorical Variables ---
    if categorical_cols_for_analysis_filtered:
        display(Markdown("##### Categorical Distributions (Bar Charts - Filtered)"))
        for col in categorical_cols_for_analysis_filtered:
            if col in df_analysis.columns:
                # Clean up column name for title
                clean_col_name = col.replace('_Label','').replace('_',' ')
                try:
                    # Adjust figure height based on number of categories
                    num_categories = df_analysis[col].nunique(dropna=False)
                    # Check if data exists for the column before plotting
                    if df_analysis[col].notna().any() and num_categories > 0:
                        dynamic_height = max(5, num_categories * 0.45)
                        plt.figure(figsize=(10, dynamic_height))

                        # Order bars by frequency
                        order = df_analysis[col].value_counts(dropna=False).index
                        ax = sns.countplot(y=df_analysis[col], order=order, palette=uni_palette)
                        ax.set_title(f'Distribution of {clean_col_name} (Austrian Academics)')
                        ax.set_xlabel('Frequency Count')
                        ax.set_ylabel('') # Y-label is clear from ticks
                        # Improve layout for long category names on y-axis
                        plt.yticks(fontsize=9)
                        plt.tight_layout()
                        plt.show()
                    else:
                        pass
                except Exception as e:
                    print(f"Could not plot categorical variable {col}. Error: {e}")
                    if plt.gcf().get_axes(): plt.close()
            else:
                print(f"Warning: Column {col} not found for plotting in filtered data.")
    else:
        print("No labeled categorical columns found for plotting in filtered data.")

else:
    print("Filtered DataFrame is empty, univariate visualizations skipped.")

### 5.2 Multivariate Analysis (Correlations & Pairplots - Filtered Data)

Explore relationships between numerical variables within the Austrian academic subset.

#### 5.2.1 Scatter Plot Matrices (Pairplots - Filtered Data)

In [ ]:
if not df_analysis.empty:
    display(Markdown("##### Scatter Plot Matrices (Pairplots - Austrian Academics)"))

    # Use the final numeric list for the filtered data
    all_numeric_vars_for_plots_filtered = numeric_cols_for_analysis_filtered

    # --- Pairplot 1: General Work Conditions & Demographics vs. Burnout ---
    display(Markdown("###### Pairplot 1: Work Conditions & Demographics vs. Burnout (Austrian Academics)"))
    pairplot1_vars_candidates = [
        'Age', 'Burnout_Score', 'Job_Satisfaction', 'Salary/hour', 'Avg_Work_Hours_HE',
        'Perceived_Autonomy', 'Performance_Pressure', 'Academic_Resources',
        'Quality_Leadership', 'Sense_Community'
    ]
    pairplot1_vars = [var for var in pairplot1_vars_candidates if var in all_numeric_vars_for_plots_filtered]

    # --- Pairplot 2: Work Attitudes (VB_) vs. Burnout ---
    display(Markdown("###### Pairplot 2: Work Attitudes (VB_) vs. Burnout (Austrian Academics)"))
    pairplot2_vars_candidates = [col for col in all_numeric_vars_for_plots_filtered if col.startswith('VB_')] + ['Burnout_Score']
    pairplot2_vars = [var for var in pairplot2_vars_candidates if var in all_numeric_vars_for_plots_filtered]

    # --- Pairplot 3: Work Motivations (WM_) vs. Burnout ---
    display(Markdown("###### Pairplot 3: Work Motivations (WM_) vs. Burnout (Austrian Academics)"))
    pairplot3_vars_candidates = [col for col in all_numeric_vars_for_plots_filtered if col.startswith('WM_')] + ['Burnout_Score']
    pairplot3_vars = [var for var in pairplot3_vars_candidates if var in all_numeric_vars_for_plots_filtered]

    # Function to generate pairplot (redefined for clarity and label rotation)
    def generate_pairplot_filtered(data, vars_list, title_suffix):
        if len(vars_list) > 1:
            print(f"Generating pairplot for: {vars_list} ({len(vars_list)} variables)")
            pairplot_data = data[vars_list].copy()
            try:
                g = sns.pairplot(pairplot_data.dropna(), # Drop rows with NaN in selected vars only for pairplot
                               diag_kind='kde', # Kernel density estimate on diagonals
                               plot_kws={'alpha': 0.4, 's': 30, 'edgecolor': None}, # Scatter plot settings
                               height=1.8) # Adjust height of individual subplots

                # Rotate x-axis labels (variable names) on the bottom row
                num_plot_vars = len(vars_list)
                for i in range(num_plot_vars):
                    if i < g.axes.shape[1]: # Check column index bounds
                       g.axes[num_plot_vars-1, i].set_xlabel(g.x_vars[i], rotation=45, ha='right', va='top', fontsize=8)
                # Set y-axis labels (variable names) on the first column to default (vertical)
                for i in range(num_plot_vars):
                    if i < g.axes.shape[0]: # Check row index bounds
                        g.axes[i, 0].set_ylabel(g.y_vars[i], fontsize=8)

                plt.suptitle(f'Scatter Plot Matrix: {title_suffix} (Austrian Academics)', y=1.02, fontsize=14)
                g.fig.tight_layout(rect=[0, 0.02, 1, 0.98]) # Adjust layout
                plt.show()
            except Exception as e:
                print(f"Could not generate pairplot for {title_suffix}. Error: {e}")
                if plt.gcf().get_axes(): plt.close()
        else:
            print(f"Skipping pairplot for {title_suffix}: Not enough variables ({len(vars_list)} found).")

    # Generate the plots for the filtered data
    generate_pairplot_filtered(df_analysis, pairplot1_vars, "Work Conditions & Demographics vs. Burnout")
    generate_pairplot_filtered(df_analysis, pairplot2_vars, "Work Attitudes (VB_) vs. Burnout")
    generate_pairplot_filtered(df_analysis, pairplot3_vars, "Work Motivations (WM_) vs. Burnout")

else:
    print("Filtered DataFrame is empty, pairplots skipped.")

#### 5.2.2 Full Correlation Matrix (Pearson - Filtered Data)

Visualize the linear relationships between all numerical variables in the Austrian academic subset.

In [ ]:
if not df_analysis.empty:
    display(Markdown("##### Full Correlation Matrix (Pearson - Austrian Academics)"))

    # Use the final list of numeric variables for the filtered data
    numeric_cols_for_corr_filtered = numeric_cols_for_analysis_filtered

    if not numeric_cols_for_corr_filtered.empty and len(numeric_cols_for_corr_filtered) > 1:
        # Calculate Pearson correlation matrix
        correlation_matrix_filtered = df_analysis[numeric_cols_for_corr_filtered].corr(method='pearson')

        # Create mask for the upper triangle
        mask = np.triu(np.ones_like(correlation_matrix_filtered, dtype=bool))

        # Set up and display the heatmap
        plt.figure(figsize=(max(12, len(numeric_cols_for_corr_filtered)*0.6), max(10, len(numeric_cols_for_corr_filtered)*0.5)))
        sns.heatmap(correlation_matrix_filtered,
                    mask=mask,
                    cmap='coolwarm', vmax=1, vmin=-1, center=0,
                    linewidths=.5, cbar_kws={"shrink": .7}, annot=False, fmt=".2f")
        plt.title('Full Correlation Matrix (Pearson - Austrian Academics)', fontsize=16)
        plt.xticks(rotation=60, ha='right', fontsize=9)
        plt.yticks(rotation=0, fontsize=9)
        plt.tight_layout()
        plt.show()
    elif len(numeric_cols_for_corr_filtered) <= 1:
        print("Not enough quantitative columns (>1) to calculate correlation matrix in filtered data.")
    else:
        print("No quantitative columns found for correlation analysis in filtered data.")
else:
    print("Filtered DataFrame is empty, full correlation analysis skipped.")

#### 5.2.3 Grouped Correlation Matrices (Filtered Data)

Examine correlations within domains and between predictors and `Burnout_Score` for Austrian academics.

In [ ]:
# Define the heatmap plotting function here, before it's called
def plot_correlation_heatmap(corr_matrix, title):
    """Helper function to plot a correlation heatmap."""
    if corr_matrix.empty or corr_matrix.shape[0] < 1 or corr_matrix.shape[1] < 1:
        print(f"Skipping heatmap for '{title}': Not enough variables or empty matrix.")
        return
    # Determine if mask is needed (only for square matrices with >1 var)
    mask = None
    if corr_matrix.shape[0] == corr_matrix.shape[1] and corr_matrix.shape[0] > 1:
         mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    annot_size = 8 if max(corr_matrix.shape) < 15 else 7 # Adjust annotation size
    plt.figure(figsize=(max(8, corr_matrix.shape[1]*0.8), max(6, corr_matrix.shape[0]*0.6)))
    sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', vmax=1, vmin=-1, center=0,
                linewidths=.5, cbar_kws={"shrink": .7}, annot=True, fmt=".2f", annot_kws={"size": annot_size})
    plt.title(title, fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.show()

# Now proceed with the rest of the code that calls the function
if not df_analysis.empty:
    display(Markdown("##### Grouped Correlation Matrices (Austrian Academics)"))

    # Use the final numeric list for the filtered data
    all_numeric_vars_corr_filtered = numeric_cols_for_analysis_filtered

    # --- Define Variable Groups (using final numeric variable names) ---
    demographic_vars_num_f = [v for v in ['Age'] if v in all_numeric_vars_corr_filtered]
    work_conditions_vars_num_f = [v for v in ['Avg_Work_Hours_HE', 'Salary/hour', 'Salary effort/hour',
                                             'Avg_Work_Hours_Other', 'Teaching %', 'Research %',
                                             'Activities related to externally funded research projects %',
                                             'Organisational and administrative activities %'] if v in all_numeric_vars_corr_filtered]
    subjective_perception_vars_f = [v for v in ['Academic_Resources', 'Performance_Pressure', 'Perceived_Autonomy',
                                                 'Quality_Leadership', 'Sense_Community'] if v in all_numeric_vars_corr_filtered]
    motivation_vars_f = [v for v in all_numeric_vars_corr_filtered if v.startswith('WM_')]
    attitudes_vars_f = [v for v in all_numeric_vars_corr_filtered if v.startswith('VB_')]
    wellbeing_outcomes_f = [v for v in ['Job_Satisfaction', 'Burnout_Score', 'Vulnerability'] if v in all_numeric_vars_corr_filtered]

    # Calculate full correlation matrix for filtered data if not already done
    if 'correlation_matrix_filtered' not in locals() or correlation_matrix_filtered.empty:
        if not all_numeric_vars_corr_filtered.empty and len(all_numeric_vars_corr_filtered) > 1:
            correlation_matrix_filtered = df_analysis[all_numeric_vars_corr_filtered].corr(method='pearson')
        else:
            print("Cannot calculate correlation_matrix_filtered: Not enough numeric variables.")
            correlation_matrix_filtered = pd.DataFrame()

    # --- Plot Correlations Within Groups ---
    if not correlation_matrix_filtered.empty:
        groups_to_plot_intra_f = {
            "Work Conditions (Numeric)": work_conditions_vars_num_f,
            "Subjective Perceptions": subjective_perception_vars_f,
            "Motivations (WM_)": motivation_vars_f,
            "Attitudes (VB_)": attitudes_vars_f,
            "Well-being Outcomes": wellbeing_outcomes_f
        }

        for group_name, group_vars in groups_to_plot_intra_f.items():
            valid_group_vars = [var for var in group_vars if var in all_numeric_vars_corr_filtered]
            if len(valid_group_vars) > 1:
                group_corr = correlation_matrix_filtered.loc[valid_group_vars, valid_group_vars]
                plot_correlation_heatmap(group_corr, f'Intra-Group Correlation: {group_name} (Austrian Academics)')
            else:
                print(f"Skipping intra-group heatmap for '{group_name}': Not enough valid variables (>1 required). Found: {valid_group_vars}")

        # --- Plot Correlations Between Predictors and Key Outcome (Burnout_Score) ---
        key_outcome_burnout_f = ['Burnout_Score']
        valid_key_outcome_burnout_f = [var for var in key_outcome_burnout_f if var in all_numeric_vars_corr_filtered]

        predictor_groups_numeric_f = {
            "Demographics & Work Conditions": demographic_vars_num_f + work_conditions_vars_num_f,
            "Subjective Perceptions": subjective_perception_vars_f,
            "Motivations (WM_)": motivation_vars_f,
            "Attitudes (VB_)": attitudes_vars_f
        }

        if valid_key_outcome_burnout_f:
            display(Markdown(f"#### Correlations between Predictor Groups and {valid_key_outcome_burnout_f[0]} (Austrian Academics)"))
            for group_name, pred_vars in predictor_groups_numeric_f.items():
                valid_pred_vars = [var for var in pred_vars if var in all_numeric_vars_corr_filtered]
                if valid_pred_vars:
                    if valid_key_outcome_burnout_f[0] in correlation_matrix_filtered.columns:
                        cross_corr = correlation_matrix_filtered.loc[valid_pred_vars, valid_key_outcome_burnout_f]
                        if not cross_corr.empty:
                            plot_correlation_heatmap(cross_corr, f'Correlations: {group_name} vs. {valid_key_outcome_burnout_f[0]} (Austrian Academics)')
                        else:
                            print(f"No valid correlations found for '{group_name}' vs. {valid_key_outcome_burnout_f[0]}.')")
                    else:
                        print(f"Key outcome '{valid_key_outcome_burnout_f[0]}' not found in correlation matrix columns for group '{group_name}'.")
                else:
                    print(f"Skipping cross-correlation for '{group_name}': No valid predictor variables found.")
        else:
            print(f"Key outcome variable ({key_outcome_burnout_f[0]}) not found or not numeric in filtered data.")
    else:
        print("Correlation matrix for filtered data is empty, skipping grouped analysis.")

else:
    print("Filtered DataFrame is empty, grouped correlation analysis skipped.")

### 5.3 Bivariate Analysis: Categorical vs. Numerical Variables (Filtered Data)

Visualize distributions of numerical variables across categories within the Austrian academic subset.

In [ ]:
if 'df_analysis' in locals() and not df_analysis.empty:

    # Use the numeric and categorical lists identified for the filtered data
    numeric_cols_for_biv_plots_f = numeric_cols_for_analysis_filtered
    categorical_cols_for_biv_plots_f = categorical_cols_for_analysis_filtered

    biv_palette = "pastel"

    if not numeric_cols_for_biv_plots_f.empty and categorical_cols_for_biv_plots_f:
        display(Markdown("##### Comparison of Numerical Distributions by Category (Austrian Academics)"))

        for cat_col in categorical_cols_for_biv_plots_f:
            if cat_col in df_analysis.columns:
                clean_cat_col_name = cat_col.replace('_Label','').replace('_',' ')
                display(Markdown(f"###### Comparisons by: {clean_cat_col_name}"))

                for num_col in numeric_cols_for_biv_plots_f:
                    clean_num_col_name = num_col.replace('_', ' ')
                    try:
                        num_categories = df_analysis[cat_col].nunique(dropna=False)
                        if df_analysis[[cat_col, num_col]].dropna().empty or num_categories == 0:
                            continue

                        fig_height = 6
                        max_label_len = 0
                        if df_analysis[cat_col].dtype == 'category':
                            if df_analysis[cat_col].cat.categories.inferred_type == 'string':
                                max_label_len = df_analysis[cat_col].cat.categories.str.len().max()
                        elif df_analysis[cat_col].dtype == 'object':
                            if df_analysis[cat_col].dropna().apply(lambda x: isinstance(x, str)).all():
                                max_label_len = df_analysis[cat_col].str.len().max()

                        base_width_per_cat = 0.8
                        if max_label_len > 15:
                            base_width_per_cat = max_label_len * 0.08
                        fig_width = max(10, num_categories * base_width_per_cat)

                        plt.figure(figsize=(fig_width, fig_height))

                        plot_order = None
                        if isinstance(df_analysis[cat_col].dtype, pd.CategoricalDtype):
                            if df_analysis[cat_col].cat.ordered:
                                plot_order = df_analysis[cat_col].cat.categories.tolist()
                            elif num_categories < 15:
                                plot_order = df_analysis[cat_col].value_counts().index

                        ax = sns.violinplot(x=cat_col, y=num_col, data=df_analysis, palette=biv_palette,
                                            cut=0, inner='quartile', order=plot_order)

                        ax.set_title(f'{clean_num_col_name} by {clean_cat_col_name} (Austrian Academics)')
                        ax.set_xlabel(clean_cat_col_name)
                        ax.set_ylabel(clean_num_col_name)

                        if num_categories > 4 or max_label_len > 10:
                            plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
                        else:
                            plt.setp(ax.get_xticklabels(), rotation=0)

                        plt.tight_layout()
                        plt.show()
                    except Exception as e:
                        print(f"Could not plot {num_col} by {cat_col}. Error: {e}")
                        if plt.gcf().get_axes(): plt.close()
            else:
                print(f"Warning: Categorical column {cat_col} not found in filtered data.")
    else:
        print("Not enough numerical or categorical variables for bivariate analysis in filtered data.")
else:
    print("Filtered DataFrame is empty, Bivariate Categorical vs Numerical analysis skipped.")

### 5.4 Disparity Analysis (Compact Subplots - Filtered Data)

Explore how key well-being variables differ across specific demographic and institutional groups within the Austrian academic subset. Use subplots to present comparisons more compactly.

In [ ]:
if 'df_analysis' in locals() and not df_analysis.empty:

    # --- Variables for Grouping and Target Metrics ---
    grouping_vars_labels_f = categorical_cols_for_analysis_filtered

    target_vars_grouped_candidates = [
        'Burnout_Score', 'Job_Satisfaction', 'Perceived_Autonomy',
        'Performance_Pressure', 'Academic_Resources', 'WM_Intrinsic_Motivation',
        'VB_Emotional_Distancing', 'Salary/hour'
    ]

    valid_target_vars_grouped_f = [var for var in target_vars_grouped_candidates if var in numeric_cols_for_analysis_filtered]
    print(f"Key variables selected for compact disparity analysis: {valid_target_vars_grouped_f}")

    grouped_palette = "muted"
    num_targets = len(valid_target_vars_grouped_f)

    # --- Generate Grouped Statistics and Compact Plots ---
    if grouping_vars_labels_f and valid_target_vars_grouped_f:
        for group_var in grouping_vars_labels_f:
            if group_var in df_analysis.columns:
                if df_analysis[group_var].notna().any() and df_analysis[group_var].nunique() > 1:
                    clean_group_var_name = group_var.replace('_Label','').replace('_',' ')
                    display(Markdown(f"#### Disparity Analysis by: {clean_group_var_name} (Austrian Academics)"))

                    try:
                        use_observed = pd.__version__ >= '1.5.0' and pd.api.types.is_categorical_dtype(df_analysis[group_var])
                        grouped_stats = df_analysis.groupby(group_var, observed=use_observed)[valid_target_vars_grouped_f].agg(['mean', 'median'])
                        display(grouped_stats.round(2))
                    except Exception as e:
                        print(f"Could not calculate grouped stats for {group_var}. Error: {e}")
                        continue

                    num_categories = df_analysis[group_var].nunique(dropna=False)
                    if num_categories == 0:
                        continue

                    ncols = 2 if num_targets > 1 else 1
                    nrows = math.ceil(num_targets / ncols)

                    max_cat_label_len = 0
                    if df_analysis[group_var].dtype == 'category':
                        if df_analysis[group_var].cat.categories.inferred_type == 'string':
                            max_cat_label_len = df_analysis[group_var].cat.categories.str.len().max()
                    elif df_analysis[group_var].dtype == 'object':
                        if df_analysis[group_var].dropna().apply(lambda x: isinstance(x, str)).all():
                            max_cat_label_len = df_analysis[group_var].str.len().max()

                    base_width_per_cat_subplot = 0.8
                    if max_cat_label_len > 15:
                        base_width_per_cat_subplot = max_cat_label_len * 0.07
                    elif max_cat_label_len > 10:
                        base_width_per_cat_subplot = max_cat_label_len * 0.09

                    fig_height_subplot = 5 * nrows
                    fig_width_subplot = max(10, num_categories * base_width_per_cat_subplot) * ncols

                    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_width_subplot, fig_height_subplot), squeeze=False)
                    axes = axes.flatten()

                    plot_order = None
                    if isinstance(df_analysis[group_var].dtype, pd.CategoricalDtype):
                        if df_analysis[group_var].cat.ordered:
                            plot_order = df_analysis[group_var].cat.categories.tolist()
                        elif num_categories < 15:
                            plot_order = df_analysis[group_var].value_counts().index

                    for i, target_var in enumerate(valid_target_vars_grouped_f):
                        ax = axes[i]
                        clean_target_var_name = target_var.replace('_',' ')
                        try:
                            sns.violinplot(x=group_var, y=target_var, data=df_analysis, palette=grouped_palette,
                                           cut=0, inner='quartile', order=plot_order, ax=ax)
                            ax.set_title(f'{clean_target_var_name}')
                            ax.set_xlabel('')
                            ax.set_ylabel(clean_target_var_name)
                            if num_categories > 4 or max_cat_label_len > 10:
                                plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
                            else:
                                plt.setp(ax.get_xticklabels(), rotation=0)
                        except Exception as e:
                            print(f"Could not plot {target_var} by {group_var} on subplot. Error: {e}")
                            ax.set_title(f'{clean_target_var_name} (Plot Error)')

                    for j in range(i + 1, len(axes)):
                        fig.delaxes(axes[j])

                    fig.suptitle(f'Distribution of Key Metrics by {clean_group_var_name} (Austrian Academics)', fontsize=16, y=1.02)
                    plt.tight_layout(rect=[0, 0, 1, 0.98])
                    plt.show()
            else:
                print(f"Warning: Grouping variable '{group_var}' not found in the filtered DataFrame.")
    else:
        print("Could not perform disparity analysis: Missing grouping variables or target variables in filtered data.")
else:
    print("Filtered DataFrame is empty, disparity analysis skipped.")

## 6. Preliminary EDA Conclusions and Next Steps for SEM (Austrian Academics)

This exploratory analysis, focused specifically on **Austrian academic staff**, provides crucial groundwork for the subsequent Structural Equation Modeling (SEM) aimed at explaining `Burnout_Score` within this subgroup.

**Key Observations Relevant to SEM (Austrian Academics):**
* **Variable Distributions:** The univariate analysis revealed the distributions of predictors and `Burnout_Score` specific to this group. Note any skewness or kurtosis that might influence SEM estimation choices (e.g., using robust estimators like MLR).
* **Correlations & Relationships:**
    * Correlation matrices and pairplots specific to Austrian academics highlight the strength and direction of relationships relevant to the SEM (Conditions -> Motivations, Motivations -> Burnout, Conditions -> Burnout, Attitudes -> Burnout/other paths).
    * Visual inspection of pairplots for WM_ vs. Burnout and VB_ vs. Burnout provides initial support (or lack thereof) for the hypothesized mediation and moderation roles within this specific population.
* **Potential Multicollinearity:** Check the intra-group correlation heatmaps (Section 5.2.3) for very high correlations (> ~0.8) among predictors within the Austrian academic subset, which could indicate multicollinearity issues for the SEM.
* **Group Differences (Internal):** The disparity analysis (Sections 5.3 & 5.4) now shows variations *within* the Austrian academic group based on remaining demographics (Gender, Age Group) and institutional characteristics (Subject Area, Contract Duration, etc.). These factors should be considered as potential covariates or moderators in the SEM for this subgroup.
* **Key Predictor Areas:** Confirm which subjective perceptions, work attitudes (VB_), and motivations (WM_) show the strongest correlations with `Burnout_Score` among Austrian academics, guiding the emphasis in the SEM path specification.

**Next Steps Towards SEM (Austrian Academics):**
1.  **Refine Theoretical Model:** Adapt the general SEM based on the specific patterns observed in the Austrian academic data. Are all hypothesized paths supported by correlations? Are there unexpected strong relationships?
2.  **Data Preparation for SEM Software:**
    * **Use Filtered Data:** Ensure the `df_analysis` DataFrame (or a selection of its columns) is used as input.
    * **Encoding:** Convert remaining categorical variables (Gender, Age Group, Subject Area, etc.) into numerical format (dummy/effect coding) as needed by the SEM software.
    * **Scaling/Centering:** Standardize or center continuous predictors/moderators if necessary.
    * **Final Checks:** Verify data types and absence of missing values in the final analytical dataset.
3.  **SEM Estimation:** Estimate the specified model using appropriate software and a suitable estimator for the data characteristics (e.g., MLR if non-normality is a concern).
4.  **Model Evaluation:** Assess model fit (CFI, TLI, RMSEA, SRMR) and evaluate path coefficients for significance, direction, and magnitude specifically for the Austrian academic sample.
5.  **Model Modification:** Refine the model based on fit and theory, potentially testing alternative paths suggested by the EDA for this subgroup.
6.  **Mediation/Moderation Testing:** Formally test the significance of indirect (mediation via WM_) and interaction (moderation via VB_/Demographics) effects within this specific context.